# 04 — Qwen3-VL + BERT + Graph Transformer + ViT Multimodal

Notebook gồm năm phần, chạy tuần tự trong cùng một thư mục run:

1. **Cue** (mục 3): Qwen3-VL nhận ảnh có khung đỏ quanh người mục tiêu và trả về cue nhìn thấy được (người, bối cảnh, quan hệ, tình huống) cho mọi người của PAMI train, val và test. Chỉ dùng ảnh và bbox, không đọc nhãn. Mặc định nhập cue của run H100 `20261104-071259-h100`, chỉ sinh phần còn thiếu.
2. **Feature** (mục 4): backbone ảnh đóng băng (ViT-B/16, CLIP ViT-B/16 hoặc SigLIP) mã hóa ảnh người và ảnh toàn cảnh; BERT đóng băng mã hóa các node cue.
3. **Huấn luyện** (mục 5): Graph Transformer + Multimodal Transformer, ma trận biến thể × seed. Checkpoint và threshold chọn trên dev; xác suất dev và eval của từng người được lưu lại.
4. **Explanation** (mục 6): bỏ lần lượt từng modality để tạo evidence, Qwen3-VL diễn đạt evidence thành lời, rồi chấm explanation.
5. **Test một lần** (mục 7): chỉ chạy khi `RUN_TEST = True`, sau khi cấu hình đã chốt trên dev.

PAMI val được chia theo **ảnh** thành dev/eval, nên người trong cùng ảnh không rơi vào hai phần khác nhau. Dev dùng để chọn checkpoint, threshold và `k`; eval chỉ để báo cáo. Code dùng chung nằm trong `src/reasoning/`, `src/models/` và `src/datasets/validation_split.py`.

Cấu hình mặc định là **lần tái lập run H100**: nhập cue và feature của H100, rồi so số eval và test với `test_results.json` của H100 (`reproduction_check`). Chỉ đổi backbone, prompt hay biến thể khi lần tái lập khớp. Bản notebook đã chạy của run H100 (layout cũ) nằm ở `outputs/modal_executions/20261107-133419/04_VLM_Pilot.ipynb` trên volume Modal `ecer-nb04` (trong git: commit `e304c0ca`).

Qwen3-VL không được xem là nguồn bằng chứng cho quyết định của emotion model. Modality ablation bằng token 0 là intervention sơ bộ, chưa thay region attribution, Integrated Gradients hay relation ablation.

## 1. Mount Drive, đồng bộ mã nguồn và cài dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition"
)
REPO_URL = (
    "https://github.com/minh-1105/"
    "Explainable-Contextual-Emotion-Recognition-via-"
    "Text-Guided-Chain-of-Thought-Reasoning.git"
)
CODE_ROOT = Path("/content/contextual-emotion-reasoning")

assert DRIVE_ROOT.is_dir(), f"Không tìm thấy project trên Drive: {DRIVE_ROOT}"

if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")],
    check=True,
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Drive root:", DRIVE_ROOT)
print("Code root :", CODE_ROOT)

## 2. Cấu hình run

`RUN_ID = None` tạo run mới trong `outputs/multimodal_pipeline/`; điền run id cũ để tiếp tục. Khi tiếp tục, config (trừ `git_commit`) và prompt phải trùng run cũ. Đổi cấu hình là tạo run mới; cue của run cũ dùng lại được qua `cue_import_from`.

Biến thể:

| Tên | Node cue | Token bị bỏ | Node bị bỏ | Tên ở run H100 |
|---|---|---|---|---|
| `full` | cue | — | — | `full` |
| `vit_only` | cue | graph | — | `vit_only` |
| `no_person_image` | cue | person | — | `no_person` |
| `no_context_image` | cue | context | — | `no_context` |
| `no_person_cues` | cue | — | person | (mới) |
| `no_context_cues` | cue | — | context | (mới) |
| `full_no_affect` | cue đã xóa từ chỉ cảm xúc | — | — | `full_no_affect` |

Token bị bỏ được thay bằng 0 cả khi train lẫn khi chấm. Node bị bỏ được che bằng mask của Graph Transformer, tương đương xóa node; node situation luôn được giữ. Threshold được chọn trên dev theo hai lưới: `legacy` (0,05–0,95, như run H100) và `extended` (từ 1e-8, như notebook 05).

`feature_import_from` chỉ dùng được với ViT-B/16, và khi mọi cue đều được nhập nguyên từ run H100.

In [ ]:
import csv
import gc
import json
import shutil
import time
import zipfile

import numpy as np
import torch
import transformers
from tqdm.auto import tqdm

from src.datasets.emotic_dataset import EMOTION_LABELS, EmoticDataset
from src.datasets.validation_split import split_by_image
from src.evaluation.emotion_metrics import (
    constant_top_k_predictions, multilabel_metrics, paired_image_bootstrap, select_constant_top_k,
    select_per_class_thresholds, threshold_boundary_report,
)
from src.models.feature_cache import (
    H100_IMAGE_FILES, H100_NODE_FILES, IMAGE_KEYS, NODE_KEYS, image_cache_path, import_rows,
    load_features, node_cache_path, save_features,
)
from src.models.frozen_encoders import FrozenImageEncoder, encode_node_lists, encode_person_context
from src.models.multimodal_fusion import (
    build_model, drop_node_kinds, make_loader, positive_weight, predict_probabilities, train_fusion,
)
from src.reasoning.grounded_explanations import (
    EXPLANATION_PROMPT, MODALITIES, build_evidence_packages, explanation_scores, generate_explanations,
)
from src.reasoning.qwen_cues import (
    build_cue_prompt, cue_record, cue_stats, mark_target, read_jsonl, semantic_nodes, strip_affect,
)
from src.utils.run_records import (
    check_or_write_json, open_run, read_json, record_environment, sha256, write_json,
)

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("transformers:", transformers.__version__)
print("git commit:", GIT_COMMIT)

EMOTIC_ROOT = DRIVE_ROOT / "data/raw/raw/emotic"
PAMI_ROOT = EMOTIC_ROOT / "PAMI"
CVPR_ROOT = EMOTIC_ROOT / "CVPR"
ANNOTATION_PATH = PAMI_ROOT / "Annotations/Annotations/Annotations.mat"

LOCAL_EMOTIC_ROOT = Path("/content/emotic")
IMAGE_ARCHIVES = (
    (PAMI_ROOT / "pami_emotic.zip", LOCAL_EMOTIC_ROOT / "PAMI", "emotic", 23185),
    (CVPR_ROOT / "cvpr_emotic.zip", LOCAL_EMOTIC_ROOT / "CVPR", "cvpr_emotic", 18302),
)
H100_RUN = "outputs/qwen3_multimodal/20261104-071259-h100"


def extract_images(archive_path, extract_dir, root_name, expected_files):
    """Giải nén zip ảnh từ Drive vào ổ local của Colab, mỗi runtime một lần."""
    image_root = extract_dir / root_name
    marker = extract_dir / ".extracted"
    if marker.is_file():
        return image_root
    assert archive_path.is_file(), f"Thiếu file zip: {archive_path}"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    started = time.time()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    file_count = sum(1 for path in image_root.rglob("*") if path.is_file())
    assert file_count == expected_files, (
        f"{archive_path.name}: {file_count} file, cần {expected_files}"
    )
    marker.touch()
    print(f"Giải nén {archive_path.name}: {file_count:,} file, {time.time() - started:.0f}s")
    return image_root


assert ANNOTATION_PATH.is_file(), f"Thiếu annotation: {ANNOTATION_PATH}"
IMAGE_ROOTS = tuple(extract_images(*spec) for spec in IMAGE_ARCHIVES)
ENVIRONMENT = {
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}

assert torch.cuda.is_available(), "Hãy bật GPU: Runtime > Change runtime type > GPU"

LEGACY_THRESHOLD_GRID = np.arange(0.05, 1.0, 0.05).tolist()  # Mặc định của select_per_class_thresholds.
EXTENDED_THRESHOLD_GRID = (
    [m * 10.0 ** e for e in range(-8, -2) for m in (1, 3)] + [0.01, 0.02, 0.03, 0.04] + LEGACY_THRESHOLD_GRID
)

PIPELINE_CONFIG = {
    # Cue (mục 3)
    "qwen_model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "max_pixels": 640 * 640,
    "max_new_tokens_cues": 192,
    "max_person_cues": 3,
    "max_context_cues": 3,
    "max_relations": 2,
    "cue_import_from": H100_RUN,  # Run H100 hoặc một run cũ của notebook này; None: sinh toàn bộ.
    # Feature (mục 4)
    "image_backbone": "vit_b_16_imagenet1k_v1",  # hoặc "clip_vit_b_16", "siglip_base_patch16_224"
    "bert_model_id": "google-bert/bert-base-uncased",
    "bert_max_length": 32,
    "max_graph_nodes": 9,
    "feature_import_from": H100_RUN,  # None: tự mã hóa toàn bộ feature.
    # Huấn luyện (mục 5)
    "category_policy": "union",
    "split_seed": 42,
    "validation_dev_fraction": 0.50,
    "fusion_dim": 256,
    "graph_layers": 2,
    "fusion_layers": 2,
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "pos_weight_max": 10.0,
    "seeds": [42, 43, 44],
    "variants": {
        "full": {"nodes": "cues", "drop_token": None, "drop_node_types": []},
        "vit_only": {"nodes": "cues", "drop_token": "graph", "drop_node_types": []},
        "no_person_image": {"nodes": "cues", "drop_token": "person", "drop_node_types": []},
        "no_context_image": {"nodes": "cues", "drop_token": "context", "drop_node_types": []},
        "no_person_cues": {"nodes": "cues", "drop_token": None, "drop_node_types": ["person"]},
        "no_context_cues": {"nodes": "cues", "drop_token": None, "drop_node_types": ["context"]},
        "full_no_affect": {"nodes": "affect_free", "drop_token": None, "drop_node_types": []},
    },
    "threshold_grids": {"legacy": LEGACY_THRESHOLD_GRID, "extended": EXTENDED_THRESHOLD_GRID},
    # Explanation (mục 6)
    "explain_variant": "full",
    "explain_seed": 42,
    "explain_threshold_grid": "legacy",
    "max_new_tokens_explanation": 256,
    "explanation_batch_size": 32,
    "explanation_reuse_from": H100_RUN,  # None: sinh toàn bộ explanation.
    # Test (mục 7)
    "primary_variant": "full",
    "bootstrap_grid": "extended",
    "bootstrap_resamples": 2000,
    "bootstrap_seed": 42,
    "resnet_run": "outputs/resnet_evaluation/20261101-034030",
    "git_commit": GIT_COMMIT,
}
RUN_ID = None  # Ví dụ "20261110-120000" để tiếp tục đúng run cũ.
CUE_BUDGET_HOURS = 20
BATCH_SIZE = None  # None: đo batch 16/32/48 rồi chọn batch nhanh nhất có schema đạt từ 95%.
BATCHES_PER_SYNC = 10
RUN_TEST = False  # Chỉ bật sau khi đã chốt cấu hình trên dev; mục 7 chấm test đúng một lần.
EXPLAIN_TEST = True  # Mục 7 sinh explanation cho mọi người của test.
H100_NAMES = {
    "full": "full", "vit_only": "vit_only", "no_person_image": "no_person",
    "no_context_image": "no_context", "full_no_affect": "full_no_affect",
}
VARIANTS, SEEDS = PIPELINE_CONFIG["variants"], PIPELINE_CONFIG["seeds"]
spec = VARIANTS[PIPELINE_CONFIG["explain_variant"]]
assert spec["drop_token"] is None, "Biến thể giải thích không được bỏ token khi train."

RUN_ID, RUN_DIR = open_run(DRIVE_ROOT / "outputs/multimodal_pipeline", RUN_ID, PIPELINE_CONFIG)
record_environment(RUN_DIR, GIT_COMMIT, ENVIRONMENT)
CUE_DIR, FEATURE_DIR, TRAIN_DIR, EXPLAIN_DIR, TEST_DIR = (
    RUN_DIR / name for name in ("cues", "features", "train", "explanations", "test")
)
for directory in (CUE_DIR, FEATURE_DIR, TRAIN_DIR / "checkpoints", TRAIN_DIR / "predictions",
                  TRAIN_DIR / "history", EXPLAIN_DIR, TEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CUE_PROMPT = build_cue_prompt(
    PIPELINE_CONFIG["max_person_cues"], PIPELINE_CONFIG["max_context_cues"], PIPELINE_CONFIG["max_relations"]
)
check_or_write_json(RUN_DIR / "prompts.json", {"cue_extraction": CUE_PROMPT, "grounded_explanation": EXPLANATION_PROMPT})

datasets = {
    split: EmoticDataset(
        annotation_path=ANNOTATION_PATH, split=split, image_roots=IMAGE_ROOTS,
        category_policy=PIPELINE_CONFIG["category_policy"], strict_images=True,
    )
    for split in ("val", "train", "test")
}
sample_ids = {split: [sample.sample_id for sample in dataset.samples] for split, dataset in datasets.items()}
index_by_id = {
    sample_id: (split, index) for split, ids in sample_ids.items() for index, sample_id in enumerate(ids)
}
print("Run dir:", RUN_DIR)
print({split: len(ids) for split, ids in sample_ids.items()})

### 2b. Chia val thành dev/eval theo ảnh

Cùng thuật toán với run H100. Với seed 42 và tỉ lệ 0,5, split phải trùng `samples.json` của H100, để mọi so sánh dùng đúng cùng một tập người.

In [ ]:
samples = split_by_image(
    datasets["train"], datasets["val"], PIPELINE_CONFIG["split_seed"], PIPELINE_CONFIG["validation_dev_fraction"]
)
check_or_write_json(RUN_DIR / "samples.json", samples)
h100_samples = DRIVE_ROOT / H100_RUN / "samples.json"
if h100_samples.is_file():
    same_split = read_json(h100_samples) == samples
    print("Split trùng run H100:", same_split)
    if (PIPELINE_CONFIG["split_seed"], PIPELINE_CONFIG["validation_dev_fraction"]) == (42, 0.5):
        assert same_split, "Split khác run H100 dù cùng seed và tỉ lệ; kiểm tra annotation."
print(samples["counts"])

## 3. Cue Qwen3-VL cho train, val và test

### 3a. Nhập cue từ run nguồn

Chạy một lần cho mỗi run. Model, giới hạn cue và prompt phải trùng run nguồn. Bản ghi trùng `sample_id` giữ bản sau cùng, như run H100 đã đọc. `cues/source_artifacts.json` ghi hash file nguồn, số người được nhập theo split và số bản ghi trùng.

In [ ]:
cue_paths = {split: CUE_DIR / f"cues_{split}.jsonl" for split in sample_ids}
manifest_path = CUE_DIR / "source_artifacts.json"
if PIPELINE_CONFIG["cue_import_from"] is not None and not manifest_path.is_file():
    source_dir = DRIVE_ROOT / PIPELINE_CONFIG["cue_import_from"]
    source_config = read_json(source_dir / "config.json")
    for key in ("qwen_model_id", "max_pixels", "max_new_tokens_cues", "max_person_cues",
                "max_context_cues", "max_relations"):
        assert source_config[key] == PIPELINE_CONFIG[key], f"{key} khác run nguồn"
    assert read_json(source_dir / "prompts.json")["cue_extraction"] == CUE_PROMPT, "Prompt khác run nguồn"
    if (source_dir / "cues").is_dir():
        source_files = sorted((source_dir / "cues").glob("cues_*.jsonl"))
    else:
        source_files = [path for path in (source_dir / "qwen_cues.jsonl", source_dir / "qwen_cues_test.jsonl")
                        if path.is_file()]
    imported, duplicates = {}, 0
    for path in source_files:
        for record in read_jsonl(path):
            duplicates += record["sample_id"] in imported
            imported[record["sample_id"]] = record
    counts = {}
    for split, ids in sample_ids.items():
        records = [imported[sample_id] for sample_id in ids if sample_id in imported]
        with cue_paths[split].open("w", encoding="utf-8") as file:
            for record in records:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")
        counts[split] = len(records)
    write_json(manifest_path, {
        "source_run": PIPELINE_CONFIG["cue_import_from"],
        "files": {path.name: {"sha256": sha256(path), "bytes": path.stat().st_size} for path in source_files},
        "imported_people": counts,
        "duplicate_records": duplicates,
        "unused_records": len(imported) - sum(counts.values()),
    })
if manifest_path.is_file():
    print(json.dumps(read_json(manifest_path), indent=2))

### 3b. Sinh cue còn thiếu

Chỉ nạp Qwen khi còn người chưa có cue; thứ tự sinh là val, train, test. Khi `BATCH_SIZE = None`, cell đo batch 16/32/48 (kết quả đo không ghi vào file cue), ghi `cues/batch_benchmark.json` và dừng nếu số giờ dự kiến vượt `CUE_BUDGET_HOURS`. Cứ `BATCHES_PER_SYNC` batch thì đóng file để Drive đồng bộ; mất kết nối thì chạy lại cell này.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from src.reasoning.qwen_vl import ask_qwen_batch, load_qwen


def missing_cues(split):
    done = {record["sample_id"] for record in read_jsonl(cue_paths[split])}
    return [sample_id for sample_id in sample_ids[split] if sample_id not in done]


todo = {split: missing_cues(split) for split in sample_ids}
remaining = sum(map(len, todo.values()))
print("Còn thiếu:", {split: len(ids) for split, ids in todo.items()})

if remaining:
    qwen_processor, qwen_model = load_qwen(PIPELINE_CONFIG["qwen_model_id"], PIPELINE_CONFIG["max_pixels"])
    image_pool = ThreadPoolExecutor(8)

    def load_marked(sample_id):
        split, index = index_by_id[sample_id]
        item = datasets[split][index]
        return item, mark_target(item["context_image"], item["bbox"].tolist())

    def run_batch(batch_ids):
        started = time.time()
        loaded = list(image_pool.map(load_marked, batch_ids))
        texts, truncated = ask_qwen_batch(
            qwen_processor, qwen_model, [image for _, image in loaded],
            [CUE_PROMPT] * len(loaded), PIPELINE_CONFIG["max_new_tokens_cues"],
        )
        seconds = round((time.time() - started) / len(batch_ids), 2)
        return [cue_record(item, raw, cut, seconds) for (item, _), raw, cut in zip(loaded, texts, truncated)]

    if BATCH_SIZE is None:
        probe = [sample_id for ids in todo.values() for sample_id in ids][:48]
        benchmark = {}
        for size in [size for size in (16, 32, 48) if size <= len(probe)] or [len(probe)]:
            torch.cuda.reset_peak_memory_stats()
            started = time.time()
            try:
                records = run_batch(probe[:size])
            except torch.cuda.OutOfMemoryError:
                print(f"batch {size}: hết VRAM")
                torch.cuda.empty_cache()
                break
            benchmark[size] = {
                "seconds_per_person": (time.time() - started) / size,
                "schema_rate": float(np.mean([record["schema_ok"] for record in records])),
                "truncated_rate": float(np.mean([record["truncated"] for record in records])),
                "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
            }
            print(f"batch {size}:", benchmark[size])
        usable = [size for size, row in benchmark.items() if row["schema_rate"] >= 0.95]
        assert usable, "Cue theo batch sai schema; dừng lại và đọc output."
        BATCH_SIZE = min(usable, key=lambda size: benchmark[size]["seconds_per_person"])
        hours = remaining * benchmark[BATCH_SIZE]["seconds_per_person"] / 3600
        write_json(CUE_DIR / "batch_benchmark.json", {
            **{str(size): row for size, row in benchmark.items()},
            "projection": {"batch_size": BATCH_SIZE, "remaining_people": remaining, "hours": hours},
        })
        print(f"Chọn BATCH_SIZE = {BATCH_SIZE}; còn {remaining:,} người, dự kiến {hours:.1f} giờ")
        assert hours <= CUE_BUDGET_HOURS, "Sinh cue không kịp trong ngân sách giờ; dừng lại và đọc output."

    progress = tqdm(total=remaining, desc="Qwen3-VL cues")
    for split, ids in todo.items():
        batches = [ids[start:start + BATCH_SIZE] for start in range(0, len(ids), BATCH_SIZE)]
        for position in range(0, len(batches), BATCHES_PER_SYNC):
            with cue_paths[split].open("a", encoding="utf-8") as file:
                for batch_ids in batches[position:position + BATCHES_PER_SYNC]:
                    for record in run_batch(batch_ids):
                        file.write(json.dumps(record, ensure_ascii=False) + "\n")
                    progress.update(len(batch_ids))
    progress.close()
    image_pool.shutdown()
    del qwen_model, qwen_processor
    gc.collect()
    torch.cuda.empty_cache()

### 3c. Kiểm tra đủ cue

Mọi người của mọi split phải có cue; nếu thiếu, chạy lại mục 3b. Tỉ lệ parse, đúng schema và bị cắt được ghi vào `cues/cue_stats.json`.

In [ ]:
cues, stats = {}, {}
for split, ids in sample_ids.items():
    records = {record["sample_id"]: record for record in read_jsonl(cue_paths[split])}
    assert set(ids) <= set(records), f"{split}: thiếu {len(set(ids) - set(records)):,} cue; chạy lại mục 3b."
    cues[split] = records
    stats[split] = cue_stats([records[sample_id] for sample_id in ids])
write_json(CUE_DIR / "cue_stats.json", stats)
print(json.dumps(stats, indent=2))

## 4. Feature ảnh và node cue

Mỗi split có một file cache trong `features/`; nếu cache đã có thì chỉ nạp lại. Mục này mã hóa train và val; test chỉ được mã hóa ở mục 7. Khi nhập feature từ H100, cell mã hóa lại 32 người đầu của val bằng code mới và ghi chênh lệch lớn nhất vào `features/feature_check.json`. Nếu chênh lệch vượt 1e-3, code mới không tái lập được feature cũ và cell dừng.

In [ ]:
IMAGE_SPEC = {
    "image_backbone": PIPELINE_CONFIG["image_backbone"],
    "category_policy": PIPELINE_CONFIG["category_policy"],
    "inputs": "bbox person crop and full image, PIL RGB",
}
NODE_SPEC = {key: PIPELINE_CONFIG[key] for key in (
    "bert_model_id", "bert_max_length", "max_graph_nodes", "max_person_cues", "max_context_cues", "max_relations",
)}
NODE_SETS = sorted({variant["nodes"] for variant in VARIANTS.values()})
NODE_LIMITS = [NODE_SPEC[key] for key in ("max_person_cues", "max_context_cues", "max_relations", "max_graph_nodes")]

feature_source = None
if PIPELINE_CONFIG["feature_import_from"] is not None:
    assert PIPELINE_CONFIG["image_backbone"] == "vit_b_16_imagenet1k_v1", "Chỉ nhập được feature ViT-B/16."
    manifest = read_json(manifest_path)
    assert manifest["source_run"] == PIPELINE_CONFIG["feature_import_from"], "Cue không nhập từ run này."
    for split, ids in sample_ids.items():
        assert manifest["imported_people"][split] == len(ids), f"Có cue {split} mới sinh; không nhập feature được."
    feature_source = DRIVE_ROOT / PIPELINE_CONFIG["feature_import_from"]


def node_lists(split, node_set, ids):
    lists = [semantic_nodes(cues[split][sample_id], *NODE_LIMITS) for sample_id in ids]
    if node_set == "affect_free":
        lists = [[(kind, strip_affect(text)) for kind, text in nodes] for nodes in lists]
    return lists


def compute_image_features(split, indices):
    encoder = FrozenImageEncoder(PIPELINE_CONFIG["image_backbone"])
    arrays = encode_person_context(
        datasets[split].__getitem__, indices, encoder,
        progress=lambda starts: tqdm(starts, desc=f"{PIPELINE_CONFIG['image_backbone']} {split}"),
    )
    encoder.close()
    return arrays


def compute_node_features(split, node_set, ids):
    return encode_node_lists(
        node_lists(split, node_set, ids), PIPELINE_CONFIG["bert_model_id"], PIPELINE_CONFIG["max_graph_nodes"],
        PIPELINE_CONFIG["bert_max_length"], progress=lambda starts: tqdm(starts, desc=f"BERT {node_set} {split}"),
    )


def split_features(split):
    """Image and node features of one PAMI split, imported or computed once, then cached."""
    ids = sample_ids[split]
    path = image_cache_path(FEATURE_DIR, PIPELINE_CONFIG["image_backbone"], split, IMAGE_SPEC)
    if not path.is_file():
        arrays = (import_rows(feature_source / H100_IMAGE_FILES[split], ids, IMAGE_KEYS) if feature_source
                  else compute_image_features(split, list(range(len(ids)))))
        save_features(path, ids, arrays)
    result = {"image": load_features(path, ids, IMAGE_KEYS)}
    for node_set in NODE_SETS:
        path = node_cache_path(FEATURE_DIR, node_set, split, NODE_SPEC)
        if not path.is_file():
            arrays = (import_rows(feature_source / H100_NODE_FILES[node_set, split], ids, NODE_KEYS) if feature_source
                      else compute_node_features(split, node_set, ids))
            save_features(path, ids, arrays)
        result[node_set] = load_features(path, ids, NODE_KEYS)
    gc.collect()
    torch.cuda.empty_cache()
    return result


features = {split: split_features(split) for split in ("train", "val")}

check_path = FEATURE_DIR / "feature_check.json"
if feature_source is not None and not check_path.is_file():
    probe = list(range(32))
    probe_ids = [sample_ids["val"][index] for index in probe]
    image = compute_image_features("val", probe)
    check = {
        f"max_abs_diff_{key}": float(np.abs(image[key] - features["val"]["image"][key][probe]).max())
        for key in ("person_features", "context_features")
    }
    check["labels_equal"] = bool(np.array_equal(image["labels"], features["val"]["image"]["labels"][probe]))
    for node_set in NODE_SETS:
        nodes = compute_node_features("val", node_set, probe_ids)
        check[f"max_abs_diff_{node_set}_nodes"] = float(
            np.abs(nodes["node_features"] - features["val"][node_set]["node_features"][probe]).max()
        )
        check[f"{node_set}_mask_types_equal"] = bool(
            np.array_equal(nodes["node_mask"], features["val"][node_set]["node_mask"][probe])
            and np.array_equal(nodes["node_types"], features["val"][node_set]["node_types"][probe])
        )
    write_json(check_path, check)
if check_path.is_file():
    check = read_json(check_path)
    print(json.dumps(check, indent=2))
    assert all(value <= 1e-3 if isinstance(value, float) else value for value in check.values()), (
        "Code mới không tái lập feature của H100; đọc features/feature_check.json."
    )
print({split: features[split]["image"]["person_features"].shape for split in features})

## 5. Huấn luyện ma trận biến thể × seed

### 5a. Tensor theo partition

`train` là toàn bộ PAMI train; `dev` và `eval` lấy từ PAMI val theo đúng thứ tự trong `samples.json`. `pos_weight` của BCE là tỉ số âm/dương trên train, kẹp trong khoảng [1, `pos_weight_max`].

In [ ]:
PARTITION_SOURCE = {"train": "train", "dev": "val", "eval": "val"}
row_of = {split: {sample_id: row for row, sample_id in enumerate(ids)} for split, ids in sample_ids.items()}
partition_rows = {
    part: torch.tensor([row_of[source][sample_id] for sample_id in samples["splits"][part]], dtype=torch.long)
    for part, source in PARTITION_SOURCE.items()
}
targets = {
    part: features[PARTITION_SOURCE[part]]["image"]["labels"][partition_rows[part].numpy()]
    for part in PARTITION_SOURCE
}
VISUAL_DIM = features["train"]["image"]["person_features"].shape[-1]
TEXT_DIM = features["train"][NODE_SETS[0]]["node_features"].shape[-1]


def feature_tensors(source, variant, rows=None):
    """(person, context, nodes, node_mask, node_types, labels) of one split for one variant."""
    nodes = source[variant["nodes"]]
    types = torch.from_numpy(nodes["node_types"]).long()
    tensors = (
        torch.from_numpy(source["image"]["person_features"]).float(),
        torch.from_numpy(source["image"]["context_features"]).float(),
        torch.from_numpy(nodes["node_features"]).float(),
        drop_node_kinds(torch.from_numpy(nodes["node_mask"]).bool(), types, variant["drop_node_types"]),
        types,
        torch.from_numpy(source["image"]["labels"]).float(),
    )
    return tensors if rows is None else tuple(tensor[rows] for tensor in tensors)


def partition_loader(part, variant):
    tensors = feature_tensors(features[PARTITION_SOURCE[part]], variant, partition_rows[part])
    return make_loader(tensors, PIPELINE_CONFIG["batch_size"], part == "train")


def load_checkpoint(name, seed):
    checkpoint = torch.load(TRAIN_DIR / "checkpoints" / f"{name}_seed{seed}.pt", map_location="cpu", weights_only=False)
    assert checkpoint["class_names"] == list(EMOTION_LABELS)
    model = build_model(PIPELINE_CONFIG, checkpoint["visual_dim"], checkpoint["text_dim"], len(EMOTION_LABELS))
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to("cuda").eval(), checkpoint


POS_WEIGHT = positive_weight(targets["train"], PIPELINE_CONFIG["pos_weight_max"])
print({part: len(values) for part, values in targets.items()}, "visual", VISUAL_DIM, "text", TEXT_DIM)

### 5b. Huấn luyện

Mỗi lần train gọi `torch.manual_seed(seed)` ngay trước khi tạo model, như run H100, nên khởi tạo và thứ tự batch tái lập được. Checkpoint là epoch có dev mAP cao nhất; threshold được chọn trên dev tại checkpoint đó. Prediction được ghi sau cùng: cặp biến thể × seed nào đã có `train/predictions/<variant>_seed<seed>.npz` là đã xong và được bỏ qua khi chạy lại.

In [ ]:
for name, variant in VARIANTS.items():
    loaders = None
    for seed in SEEDS:
        tag = f"{name}_seed{seed}"
        prediction_path = TRAIN_DIR / "predictions" / f"{tag}.npz"
        if prediction_path.is_file():
            print("Đã có", tag)
            continue
        if loaders is None:
            loaders = {part: partition_loader(part, variant) for part in PARTITION_SOURCE}
        torch.manual_seed(seed)
        model = build_model(PIPELINE_CONFIG, VISUAL_DIM, TEXT_DIM, len(EMOTION_LABELS)).to("cuda")
        state, history, best_dev_map = train_fusion(
            model, loaders, targets["dev"], epochs=PIPELINE_CONFIG["epochs"],
            learning_rate=PIPELINE_CONFIG["learning_rate"], weight_decay=PIPELINE_CONFIG["weight_decay"],
            pos_weight=POS_WEIGHT, drop_modality=variant["drop_token"], class_names=EMOTION_LABELS,
            log=lambda row: print(tag, row),
        )
        probabilities = {
            part: predict_probabilities(model, loaders[part], drop_modality=variant["drop_token"])
            for part in ("dev", "eval")
        }
        thresholds = {
            grid: select_per_class_thresholds(targets["dev"], probabilities["dev"], values)
            for grid, values in PIPELINE_CONFIG["threshold_grids"].items()
        }
        torch.save({
            "model_state_dict": {key: value.cpu() for key, value in state.items()},
            "pipeline_config": PIPELINE_CONFIG,
            "variant": name,
            "variant_spec": variant,
            "seed": seed,
            "visual_dim": VISUAL_DIM,
            "text_dim": TEXT_DIM,
            "class_names": list(EMOTION_LABELS),
            "best_dev_map": best_dev_map,
            "thresholds": {grid: values.tolist() for grid, values in thresholds.items()},
        }, TRAIN_DIR / "checkpoints" / f"{tag}.pt")
        write_json(TRAIN_DIR / "history" / f"{tag}.json", history)
        temporary = prediction_path.with_suffix(".tmp.npz")
        np.savez_compressed(
            temporary,
            dev_sample_ids=np.asarray(samples["splits"]["dev"]), dev_probabilities=probabilities["dev"],
            eval_sample_ids=np.asarray(samples["splits"]["eval"]), eval_probabilities=probabilities["eval"],
            **{f"thresholds_{grid}": values for grid, values in thresholds.items()},
        )
        temporary.replace(prediction_path)
        del model, state
        gc.collect()
        torch.cuda.empty_cache()

### 5c. Metric trên dev/eval và kiểm tra tái lập H100

Cell tính metric cho mọi biến thể × seed × lưới threshold, mean ± SD qua seed, constant top-k chọn trên dev, và số lớp có threshold chạm đáy lưới. Khi feature nhập từ H100, số eval theo lưới legacy được so với `test_results.json` của H100, theo bảng tên ở mục 2. Bootstrap ghép cặp và so sánh với ResNet trên eval làm ở notebook 05.

In [ ]:
METRICS = ("map", "macro_f1", "micro_f1", "mean_predicted_labels")
runs = []
for name in VARIANTS:
    for seed in SEEDS:
        with np.load(TRAIN_DIR / "predictions" / f"{name}_seed{seed}.npz", allow_pickle=False) as data:
            for part in ("dev", "eval"):
                assert data[f"{part}_sample_ids"].tolist() == samples["splits"][part]
            for grid, values in PIPELINE_CONFIG["threshold_grids"].items():
                boundary = threshold_boundary_report(targets["dev"], data[f"thresholds_{grid}"], values)
                for part in ("dev", "eval"):
                    result = multilabel_metrics(
                        targets[part], data[f"{part}_probabilities"], thresholds=data[f"thresholds_{grid}"],
                        class_names=EMOTION_LABELS,
                    )
                    runs.append({
                        "variant": name, "seed": seed, "grid": grid, "split": part,
                        **{metric: result[metric] for metric in METRICS},
                        "thresholds_at_grid_min": boundary["at_min"],
                    })


def mean_sd(rows, metrics):
    return {metric: {"mean": float(np.mean([row[metric] for row in rows])),
                     "sd": float(np.std([row[metric] for row in rows]))} for metric in metrics}


summary = {
    name: {
        grid: {part: mean_sd([row for row in runs if (row["variant"], row["grid"], row["split"]) == (name, grid, part)], METRICS)
               for part in ("dev", "eval")}
        for grid in PIPELINE_CONFIG["threshold_grids"]
    }
    for name in VARIANTS
}

constant = select_constant_top_k(targets["dev"])
constant_eval = multilabel_metrics(
    targets["eval"], predictions=constant_top_k_predictions(
        len(targets["eval"]), constant["label_order"], constant["k"], len(EMOTION_LABELS)
    ), class_names=EMOTION_LABELS,
)


def reproduction_check(new_rows, split):
    """Differences from the H100 runs of the same variant, seed and split (legacy grid)."""
    h100_results = DRIVE_ROOT / H100_RUN / "test_results.json"
    if feature_source is None or not h100_results.is_file():
        return None
    old_runs = read_json(h100_results)["runs"]
    rows = []
    for name, old_name in H100_NAMES.items():
        for seed in SEEDS:
            old = next(row for row in old_runs if (row["variant"], row["seed"], row["split"]) == (old_name, seed, split))
            new = next(row for row in new_rows if (row["variant"], row["seed"], row["grid"], row.get("split", split)) == (name, seed, "legacy", split))
            rows.append({"variant": name, "seed": seed, **{
                f"{metric}_diff": new[metric] - old[metric] for metric in ("map", "macro_f1", "micro_f1")
            }})
    largest = max(abs(row[key]) for row in rows for key in row if key.endswith("_diff"))
    print(f"Chênh lệch lớn nhất so với H100 trên {split}: {largest:.6f}")
    if largest > 0.005:
        print("CẢNH BÁO: không tái lập được run H100; đọc reproduction_check trước khi đi tiếp.")
    return rows


write_json(TRAIN_DIR / "metrics.json", {
    "scope": "trained on PAMI train; checkpoints and thresholds chosen on dev; eval is held-out PAMI val; no test data",
    "people": {part: len(values) for part, values in targets.items()},
    "summary": summary,
    "runs": runs,
    "constant_top_k_eval": {"k": constant["k"], "macro_f1": constant_eval["macro_f1"], "micro_f1": constant_eval["micro_f1"]},
    "reproduction_check": reproduction_check(runs, "eval"),
})
with (TRAIN_DIR / "runs.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(runs[0]))
    writer.writeheader()
    writer.writerows(runs)

for name in VARIANTS:
    row = summary[name]["extended"]["eval"]
    print(f"{name:18s} eval mAP {row['map']['mean']:.4f} ± {row['map']['sd']:.4f} | "
          f"macro-F1 (extended) {row['macro_f1']['mean']:.4f} ± {row['macro_f1']['sd']:.4f}")
print(f"constant top-{constant['k']}: macro-F1 {constant_eval['macro_f1']:.4f}")

### 5d. Biểu đồ

Dev mAP theo epoch (trung bình qua seed, vùng mờ là min–max), cùng mAP và macro-F1 trên eval (lưới extended). Hình lưu vào `train/figures/`.

In [ ]:
import matplotlib.pyplot as plt

figure_dir = TRAIN_DIR / "figures"
figure_dir.mkdir(exist_ok=True)
names = list(VARIANTS)

fig, axis = plt.subplots(figsize=(7, 4))
for name in names:
    curves = np.asarray([
        [row["dev_map"] for row in read_json(TRAIN_DIR / "history" / f"{name}_seed{seed}.json")] for seed in SEEDS
    ])
    epochs = np.arange(1, curves.shape[1] + 1)
    axis.plot(epochs, curves.mean(0), label=name)
    axis.fill_between(epochs, curves.min(0), curves.max(0), alpha=0.15)
axis.set_xlabel("epoch")
axis.set_ylabel("dev mAP")
axis.set_title(f"Dev mAP theo epoch, {len(SEEDS)} seed")
axis.legend(fontsize=8)
fig.tight_layout()
fig.savefig(figure_dir / "variant_training.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, metric, title in zip(axes, ("map", "macro_f1"), ("eval mAP", "eval macro-F1 (lưới extended)")):
    means = [summary[name]["extended"]["eval"][metric]["mean"] for name in names]
    sds = [summary[name]["extended"]["eval"][metric]["sd"] for name in names]
    axis.bar(names, means, yerr=sds, capsize=4, color="tab:green")
    axis.set_ylim(min(means) - 0.03, max(means) + 0.03)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(figure_dir / "variant_comparison.png", dpi=150)
plt.show()
print("Đã lưu hình vào", figure_dir)

## 6. Evidence can thiệp và explanation có căn cứ trên eval

### 6a. Modality ablation → evidence package

Dùng checkpoint `explain_variant`, seed `explain_seed` (mặc định `full`, seed 42, như model chính của run H100). Xác suất được tính lại phải khớp `train/predictions/`. Hai loại xác suất can thiệp được tính: `without_*` khi bỏ một modality (dùng cho comprehensiveness) và `only_*` khi chỉ giữ một modality (dùng cho sufficiency). Cả hai được lưu trong `explanations/interventions_eval.npz`.

In [ ]:
def run_interventions(model, loader, split, ids, thresholds, out_dir):
    probabilities = predict_probabilities(model, loader)
    ablated = {modality: predict_probabilities(model, loader, drop_modality=modality) for modality in MODALITIES}
    keep_only = {modality: predict_probabilities(model, loader, keep_only=modality) for modality in MODALITIES}
    np.savez_compressed(
        out_dir / f"interventions_{split}.npz", sample_ids=np.asarray(ids), probabilities=probabilities,
        thresholds=thresholds, **{f"without_{m}": v for m, v in ablated.items()},
        **{f"only_{m}": v for m, v in keep_only.items()},
    )
    return probabilities, ablated, keep_only


def explain(split, evidence, out_dir, load_image):
    """Reuse source explanations whose evidence is identical, generate the rest, return all records."""
    path = out_dir / f"explanations_{split}.jsonl"
    source = None
    if PIPELINE_CONFIG["explanation_reuse_from"] is not None:
        root = DRIVE_ROOT / PIPELINE_CONFIG["explanation_reuse_from"]
        old_name = {"eval": "explanations.jsonl", "test": "explanations_test.jsonl"}[split]
        source = next((p for p in (root / "explanations" / path.name, root / "test" / path.name, root / old_name)
                       if p.is_file()), None)
    if source is not None and not path.is_file():
        current = {package["sample_id"]: package for package in evidence}
        reused = [record for record in read_jsonl(source) if current.get(record["sample_id"]) == record["evidence"]]
        with path.open("w", encoding="utf-8") as file:
            for record in reused:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Dùng lại {len(reused):,}/{len(evidence):,} explanation có evidence trùng.")

    done = {record["sample_id"] for record in read_jsonl(path)}
    if any(package["sample_id"] not in done for package in evidence):
        from src.reasoning.qwen_vl import ask_qwen_batch, load_qwen

        processor, qwen = load_qwen(PIPELINE_CONFIG["qwen_model_id"], PIPELINE_CONFIG["max_pixels"])
        generated = generate_explanations(
            evidence, path,
            lambda images, prompts: ask_qwen_batch(
                processor, qwen, images, prompts, PIPELINE_CONFIG["max_new_tokens_explanation"]
            ),
            load_image, PIPELINE_CONFIG["explanation_batch_size"],
            progress=lambda starts: tqdm(starts, desc=f"explanations {split}"),
        )
        print("Sinh mới", generated, "explanation.")
        del qwen, processor
        gc.collect()
        torch.cuda.empty_cache()
    wanted = {package["sample_id"] for package in evidence}
    records = [record for record in read_jsonl(path) if record["sample_id"] in wanted]
    assert len(records) == len(wanted), "Chưa đủ explanation; chạy lại cell này."
    return records


def load_marked_image(split):
    def load(sample_id):
        item = datasets[split][index_by_id[sample_id][1]]
        return mark_target(item["context_image"], item["bbox"].tolist())
    return load


explain_model, explain_checkpoint = load_checkpoint(PIPELINE_CONFIG["explain_variant"], PIPELINE_CONFIG["explain_seed"])
explain_thresholds = np.asarray(explain_checkpoint["thresholds"][PIPELINE_CONFIG["explain_threshold_grid"]])
eval_ids = samples["splits"]["eval"]
eval_probabilities, eval_ablated, eval_keep_only = run_interventions(
    explain_model, partition_loader("eval", spec), "eval", eval_ids, explain_thresholds, EXPLAIN_DIR,
)
with np.load(TRAIN_DIR / "predictions" / f"{PIPELINE_CONFIG['explain_variant']}_seed{PIPELINE_CONFIG['explain_seed']}.npz") as saved:
    assert np.allclose(saved["eval_probabilities"], eval_probabilities, atol=1e-4), "Checkpoint hoặc feature đã đổi."
eval_evidence = build_evidence_packages(eval_ids, eval_probabilities, eval_ablated, explain_thresholds, cues["val"])
write_json(EXPLAIN_DIR / "evidence_packages_eval.json", eval_evidence)
print("Đã tạo", len(eval_evidence), "evidence package cho eval.")

### 6b. Qwen3-VL diễn đạt evidence và metric

Lần chạy đầu, explanation của `explanation_reuse_from` được chép sang cho những người có evidence package trùng hoàn toàn; người còn lại được sinh theo batch, file JSONL resume được. Metric giống run H100: mức khớp với modality có độ giảm lớn nhất (kèm mức khớp khi luôn chọn modality phổ biến nhất), tỉ lệ con số được trích có trong evidence, tỉ lệ nêu cảm xúc ngoài dự đoán, và comprehensiveness, sufficiency của ERASER ở mức modality. Kết quả ghi vào khóa `eval` của `explanations/explanation_metrics.json`.

In [ ]:
eval_records = explain("eval", eval_evidence, EXPLAIN_DIR, load_marked_image("val"))
metrics_path = EXPLAIN_DIR / "explanation_metrics.json"
explanation_metrics = read_json(metrics_path) if metrics_path.is_file() else {}
explanation_metrics["eval"] = explanation_scores(
    eval_records, eval_keep_only, {sample_id: index for index, sample_id in enumerate(eval_ids)}
)
write_json(metrics_path, explanation_metrics)
print(json.dumps(explanation_metrics["eval"], indent=2))

h100_metrics = DRIVE_ROOT / H100_RUN / "explanation_metrics.json"
if PIPELINE_CONFIG["explanation_reuse_from"] == H100_RUN and h100_metrics.is_file():
    old = read_json(h100_metrics)["eval"]
    for key in ("n_usable", "modality_agreement", "modality_agreement_always_majority", "cited_numbers_in_evidence"):
        print(f"{key}: H100 {old[key]} | run này {explanation_metrics['eval'][key]}")

## 7. Chấm test một lần

Chỉ bật `RUN_TEST = True` sau khi đã chốt biến thể, seed, lưới threshold và `k` trên dev, và đã so sánh trên eval ở notebook 05. Mục này không train và không chọn gì: checkpoint và threshold lấy nguyên từ mục 5. Không chỉnh cấu hình theo số test rồi chạy lại; muốn đổi gì thì tạo run mới và chọn lại trên dev.

### 7a. Metric, ResNet-50 và bootstrap

Mục này mã hóa feature test (hoặc nhập từ H100), rồi chấm mọi biến thể × seed trên 7.280 người PAMI test, cùng constant top-k chọn trên dev, mAP theo nhãn majority, và ba ResNet-50 của notebook 03 (cùng người, cùng nhãn union). Paired bootstrap theo ảnh được chạy cho từng seed, trên cả mAP và macro-F1 (`bootstrap_grid`). Khi `test/test_results.json` đã có, cell chỉ đọc lại kết quả, không chấm lại.

In [ ]:
results_path = TEST_DIR / "test_results.json"
if not RUN_TEST:
    print("RUN_TEST = False: bỏ qua mục 7.")
elif results_path.is_file():
    print("Run này đã được chấm test; chỉ đọc lại kết quả.")
    test_results = read_json(results_path)
else:
    test_ids, test_image_ids = sample_ids["test"], [sample.relative_image_path for sample in datasets["test"].samples]
    features["test"] = split_features("test")
    test_targets = features["test"]["image"]["labels"]

    probabilities, thresholds = {}, {}
    for name, variant in VARIANTS.items():
        loader = make_loader(feature_tensors(features["test"], variant), PIPELINE_CONFIG["batch_size"], False)
        for seed in SEEDS:
            model, checkpoint = load_checkpoint(name, seed)
            probabilities[name, seed] = predict_probabilities(model, loader, drop_modality=variant["drop_token"])
            thresholds[name, seed] = {grid: np.asarray(values) for grid, values in checkpoint["thresholds"].items()}
            del model
    torch.cuda.empty_cache()

    resnet_dir = DRIVE_ROOT / PIPELINE_CONFIG["resnet_run"]
    resnet_thresholds = read_json(resnet_dir / "metrics/thresholds.json")
    resnet = {}
    for name in ("person_only", "context_only", "person_context"):
        with np.load(resnet_dir / f"predictions/{name}_test.npz", allow_pickle=False) as stored:
            assert stored["sample_ids"].tolist() == test_ids
            assert np.array_equal(stored["targets_union"], test_targets.astype(np.int32))
            resnet[name] = {key: stored[key] for key in ("probabilities", "targets_majority")}
    majority_targets = resnet["person_context"]["targets_majority"]

    test_runs = []
    for (name, seed), values in probabilities.items():
        majority_map = multilabel_metrics(majority_targets, values, class_names=EMOTION_LABELS)["map"]
        for grid, chosen in thresholds[name, seed].items():
            result = multilabel_metrics(test_targets, values, thresholds=chosen, class_names=EMOTION_LABELS)
            test_runs.append({"variant": name, "seed": seed, "grid": grid,
                              **{metric: result[metric] for metric in METRICS}, "majority_map": majority_map})
    test_summary = {
        name: {grid: mean_sd([row for row in test_runs if (row["variant"], row["grid"]) == (name, grid)],
                             METRICS + ("majority_map",))
               for grid in PIPELINE_CONFIG["threshold_grids"]}
        for name in VARIANTS
    }
    constant_predictions = constant_top_k_predictions(len(test_ids), constant["label_order"], constant["k"], len(EMOTION_LABELS))
    constant_metrics = multilabel_metrics(test_targets, predictions=constant_predictions, class_names=EMOTION_LABELS)

    resnet_metrics, resnet_binary = {}, {}
    for name, stored in resnet.items():
        chosen = np.asarray(resnet_thresholds["union"][name])
        union = multilabel_metrics(test_targets, stored["probabilities"], thresholds=chosen, class_names=EMOTION_LABELS)
        majority = multilabel_metrics(majority_targets, stored["probabilities"],
                                      thresholds=resnet_thresholds["majority"][name], class_names=EMOTION_LABELS)
        resnet_metrics[name] = {**{metric: union[metric] for metric in METRICS}, "majority_map": majority["map"]}
        resnet_binary[name] = (stored["probabilities"] >= chosen).astype(np.int32)

    def bootstrap(a_scores, a_binary, b_scores, b_binary):
        options = dict(n_resamples=PIPELINE_CONFIG["bootstrap_resamples"], seed=PIPELINE_CONFIG["bootstrap_seed"])
        result = {"macro_f1": paired_image_bootstrap(test_targets, a_binary, b_binary, test_image_ids, metric="macro_f1", **options)}
        if b_scores is not None:
            result["map"] = paired_image_bootstrap(test_targets, a_scores, b_scores, test_image_ids, metric="map", **options)
        return result

    grid, primary = PIPELINE_CONFIG["bootstrap_grid"], PIPELINE_CONFIG["primary_variant"]
    binary = {key: (values >= thresholds[key][grid]).astype(np.int32) for key, values in probabilities.items()}
    comparisons = {}
    for seed in SEEDS:
        a = (probabilities[primary, seed], binary[primary, seed])
        for name in VARIANTS:
            if name != primary:
                comparisons[f"{primary}_vs_{name}_seed{seed}"] = bootstrap(*a, probabilities[name, seed], binary[name, seed])
        for name in resnet:
            comparisons[f"{primary}_vs_resnet_{name}_seed{seed}"] = bootstrap(*a, resnet[name]["probabilities"], resnet_binary[name])
        comparisons[f"{primary}_vs_constant_seed{seed}"] = bootstrap(*a, None, constant_predictions)

    np.savez_compressed(
        TEST_DIR / "predictions_test.npz", sample_ids=np.asarray(test_ids), image_ids=np.asarray(test_image_ids),
        targets_union=test_targets, targets_majority=majority_targets,
        **{f"{name}_seed{seed}_probabilities": values for (name, seed), values in probabilities.items()},
        **{f"{name}_seed{seed}_thresholds_{g}": values for (name, seed), chosen in thresholds.items() for g, values in chosen.items()},
    )
    with (TEST_DIR / "test_runs.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(test_runs[0]))
        writer.writeheader()
        writer.writerows(test_runs)
    test_results = {
        "scope": "single test-set evaluation; checkpoints, thresholds and k chosen on dev",
        "people": {"test": len(test_ids), "test_images": len(set(test_image_ids))},
        "test_cues": stats["test"],
        "summary": test_summary,
        "runs": test_runs,
        "constant_top_k": {"k": constant["k"], "map": float(test_targets.mean()),
                           "macro_f1": constant_metrics["macro_f1"], "micro_f1": constant_metrics["micro_f1"]},
        "resnet": resnet_metrics,
        "bootstrap": {"threshold_grid": grid, "comparisons": comparisons},
        "reproduction_check": reproduction_check(test_runs, "test"),
    }
    write_json(results_path, test_results)

if RUN_TEST:
    for name in VARIANTS:
        row = test_results["summary"][name][PIPELINE_CONFIG["bootstrap_grid"]]
        print(f"{name:18s} test mAP {row['map']['mean']:.4f} ± {row['map']['sd']:.4f} | "
              f"macro-F1 {row['macro_f1']['mean']:.4f} ± {row['macro_f1']['sd']:.4f}")
    for name, row in test_results["resnet"].items():
        print(f"resnet {name:12s} test mAP {row['map']:.4f} | macro-F1 {row['macro_f1']:.4f}")
    for key, result in test_results["bootstrap"]["comparisons"].items():
        print(key, {metric: f"{value['difference_a_minus_b']:+.4f} [{value['ci95_lower']:+.4f}, {value['ci95_upper']:+.4f}]"
                    for metric, value in result.items()})

### 7b. Evidence và explanation cho toàn bộ test

Chạy khi `RUN_TEST` và `EXPLAIN_TEST` đều bật. Mục này dùng đúng checkpoint, lưới threshold và prompt của mục 6. Kết quả ghi vào `test/` và vào khóa `test` của `explanations/explanation_metrics.json`.

In [ ]:
if RUN_TEST and EXPLAIN_TEST:
    if "test" not in features:
        features["test"] = split_features("test")
    test_ids = sample_ids["test"]
    test_loader = make_loader(feature_tensors(features["test"], spec), PIPELINE_CONFIG["batch_size"], False)
    test_probabilities, test_ablated, test_keep_only = run_interventions(
        explain_model, test_loader, "test", test_ids, explain_thresholds, TEST_DIR,
    )
    test_evidence = build_evidence_packages(test_ids, test_probabilities, test_ablated, explain_thresholds, cues["test"])
    write_json(TEST_DIR / "evidence_packages_test.json", test_evidence)
    test_records = explain("test", test_evidence, TEST_DIR, load_marked_image("test"))
    explanation_metrics["test"] = explanation_scores(
        test_records, test_keep_only, {sample_id: index for index, sample_id in enumerate(test_ids)}
    )
    write_json(metrics_path, explanation_metrics)
    print(json.dumps(explanation_metrics["test"], indent=2))

## 8. Bàn giao và giới hạn

- Gửi `run_id` cùng `train/metrics.json` cho team. Notebook 05 đọc `train/predictions/` để chạy bootstrap và so với ResNet trên eval.
- Lần tái lập H100 đạt khi `reproduction_check` trong `train/metrics.json` có mọi chênh lệch gần 0. Chỉ sau đó mới đổi backbone (`image_backbone`), thêm feature khuôn mặt, hoặc đổi prompt Qwen (chỉ khi ablation cho thấy cue có đóng góp).
- Cue của Qwen vẫn có thể sai; cần đánh giá thủ công cue và relation trên một mẫu cố định. Graph hiện là semantic graph nối đầy đủ, chưa phải scene graph có object và relation được định danh.
- Modality ablation bằng token 0 gây distribution shift; cần bổ sung region occlusion, Integrated Gradients và relation ablation thực sự.